# Acento Portfolio Analytics: NOI Optimization Engine
## Master Dataset Integration & Multi-Property Performance Analysis

**German Montoya | Data Scientist | Acento Partners**

---

## Step 1: Upload & Data Exploration

This notebook documents the journey from raw source files to a unified, analysis-ready dataset.

### 1.1 Load Libraries & Discover Data Files

Import the libraries we need for data manipulation and inspection, then identify and list all available source data files. We expect 3 Excel files (T12 OSD reports, one per year 2023-2025, each with property tabs and monthly data) and 1 CSV file (occupancy flat table for the same period).

In [15]:
# Data handling and file utilities
import pandas as pd
import openpyxl  # Reads .xlsx files; the T12 .xls files need pandas instead
from pathlib import Path
import json

# Show every column and wide text, so tables print without wrapping or hiding fields
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Use the folder the notebook runs from as the data folder
data_folder = Path.cwd()
print(f"Working directory: {data_folder}")
print(f"\nData files found:")

# List Excel files (.xls and .xlsx) with their size in KB
for f in sorted(data_folder.glob('*.xls*')):
    print(f"  - {f.name} ({f.stat().st_size / 1024:.0f} KB)")

# List CSV files with their size in KB
for f in sorted(data_folder.glob('*.csv')):
    print(f"  - {f.name} ({f.stat().st_size / 1024:.0f} KB)")

Working directory: /Users/germanmontoya/Python/vscode/Acento

Data files found:
  - 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-2.xls (696 KB)
  - 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-3.xls (812 KB)
  - 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties.xls (785 KB)
  - Occupancy_Summary_2025_2023.xlsx (519 KB)


**Findings**

- **4 source files:**
  - **3 T12 files** in the Acento folder (`2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties*.xls`): 2024 (no suffix), 2023 (`-2`), 2025 (`-3`)
  - **1 occupancy file:** `0ccupancy_flat_2023-2025.csv` in the Acento folder. It has 31 properties and 36 months, one row per property-month (1,116 rows).

### 1.2 Profile T12 Files

For each of the three T12 OSD files, we explore their structure to understand:
- Sheet names (which represent individual properties)
- Data layout (GL codes as rows or columns? Months as rows or columns?)
- Month range covered (Jan-Dec for each year?)
- Sign conventions (how are expenses, revenues, and NOI values signed?)
- Any subtotal rows or account hierarchy patterns

This understanding will guide how we unpivot and normalize the data into fact_long.

In [16]:
# Find the three T12 files by their shared file-name prefix
t12_files = sorted(data_folder.glob('2_by Separate tabs*.xls'))

for file_path in t12_files:
    # Open the workbook once to list its sheet names (one sheet per property)
    xls = pd.ExcelFile(file_path)
    print("=" * 80)
    print(file_path.name)
    print()
    print(f"Tabs: {len(xls.sheet_names)}")
    print(f"First 5 tabs: {xls.sheet_names[:5]}")
    print()

    # Read only the top 12 rows of each tab to find the "As of Date" value.
    # Collecting them in a set shows whether every tab in the file has the same date.
    as_of_dates = set()
    for sheet in xls.sheet_names:
        head = pd.read_excel(file_path, sheet_name=sheet, header=None, nrows=12)
        # Find the row whose first column says "As of Date" and take the value beside it
        match = head[head[0].astype(str).str.contains('As of Date', na=False)]
        as_of_dates.add(match[1].iloc[0] if len(match) else 'not found')
    print(f"As of Date(s) across all tabs: {sorted(map(str, as_of_dates))}")
    print()

    # Print one full tab (the third sheet) to see the layout in detail
    sample_sheet = xls.sheet_names[2]
    sample = pd.read_excel(file_path, sheet_name=sample_sheet, header=None, nrows=30)
    print(f"Sample tab: {sample_sheet} | shape {sample.shape}")
    print()
    print(sample.to_string())
    print()

2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-2.xls

Tabs: 31
First 5 tabs: ['Operating Statement Detail__CAL', 'Operating Statement Detail__HGA', 'Operating Statement Detail__JHA', 'Operating Statement Detail__LPA', 'Operating Statement Detail__PGA']

As of Date(s) across all tabs: ['12/31/2023']

Sample tab: Operating Statement Detail__JHA | shape (30, 14)

                                               0             1             2             3             4             5             6             7             8             9             10            11            12        13
0                                                           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN           NaN       NaN
1                                                           NaN           NaN           NaN           NaN           NaN           NaN           NaN        

**Findings**

- **3 T12 files** (Operating Statement Detail, 12-month actuals, accrual basis). The file suffixes do not indicate the year.
  - **No suffix** (`...Multifamily Properties.xls`): **2024** (as of 12/31/2024)
  - **`-2`**: **2023** (as of 12/31/2023)
  - **`-3`**: **2025** (as of 12/31/2025)
- **31 tabs per file**, one per property, named `Operating Statement Detail__[CODE]` (e.g., `__JHA` = Jefferson Hall).
- **As-of date is consistent across all tabs within each file.**
- **Header block (rows 0–9):** property name, report title, "As of" date, reporting book (ACCRUAL), and as-of date.
- **Monthly columns:** 12 month-end columns (01/31 to 12/31) plus a Total column. Months run across columns, GL accounts down rows.
- **Account rows:** GL codes with descriptions (e.g., `A4010-0100 - Market Rental Income`), plus subtotal rows (e.g., `Total Standard Rental Income`), section labels (e.g., `Summary`, `INCOME`, `Credit Provision`), and blank spacer rows.
- **Sign convention (observed in JHA, 2023–2025):** income is positive. Loss to lease and bad debt write-offs are usually negative, with exceptions (e.g., loss to lease is +3,423 in JHA 2023 December). Bad debt recoveries are usually positive, with exceptions (e.g., -498 in JHA 2025 January). Signs need to be checked across all accounts and tabs before we rely on them.

### 1.3 Check Layout Consistency

Compare every property tab against the first tab of its own file on the three things the unpivot depends on: month headers and account row labels. Month headers change by year, so they are only compared within the same file. Property names sit in the header block, so the comparison starts below it.

In [20]:
# Compare each tab only to the first tab of its own file.
# Month headers change by year, so they are never compared across files.
for file_path in t12_files:
    sheets = pd.ExcelFile(file_path).sheet_names

    # Reference tab for this file: its account labels (row 12 down) and month headers (row 11)
    ref = pd.read_excel(file_path, sheet_name=sheets[0], header=None)
    ref_labels = ref[0].iloc[12:].astype(str).str.strip().tolist()
    ref_months = ref.iloc[11, 1:13].astype(str).tolist()

    print("=" * 80)
    print(f"{file_path.name}")
    print(f"Reference: {sheets[0]} | {len(ref_labels)} label rows")
    print()

    for sheet in sheets[1:]:
        df = pd.read_excel(file_path, sheet_name=sheet, header=None)
        labels = df[0].iloc[12:].astype(str).str.strip().tolist()
        months = df.iloc[11, 1:13].astype(str).tolist()

        # Month headers must match the reference tab in the same file
        if months != ref_months:
            print(f"  {sheet}: month headers differ from reference")

        # Labels must match row for row. If they don't, show the first row that differs.
        if labels != ref_labels:
            first_diff = next(
                (i for i, (a, b) in enumerate(zip(labels, ref_labels)) if a != b),
                min(len(labels), len(ref_labels)),
            )
            # Sheet row number = list position + 12 (labels start at row 12)
            row_no = first_diff + 12
            tab_label = labels[first_diff] if first_diff < len(labels) else 'end of tab'
            ref_label = ref_labels[first_diff] if first_diff < len(ref_labels) else 'end of tab'
            print(f"  {sheet}: {len(labels)} label rows (reference {len(ref_labels)}), "
                  f"first difference at row {row_no}")
            print(f"      tab:       {tab_label}")
            print(f"      reference: {ref_label}")
    print()

2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-2.xls
Reference: Operating Statement Detail__CAL | 339 label rows

  Operating Statement Detail__HGA: 321 label rows (reference 339), first difference at row 22
      tab:       Total Gross Potential Income
      reference: Subsidized Rent
  Operating Statement Detail__JHA: 350 label rows (reference 339), first difference at row 22
      tab:       Total Gross Potential Income
      reference: Subsidized Rent
  Operating Statement Detail__LPA: 346 label rows (reference 339), first difference at row 35
      tab:       Total Credit Provision
      reference: A1200-2999 - A/R - Covid19 - Installment Payment Plan balance
  Operating Statement Detail__PGA: 321 label rows (reference 339), first difference at row 22
      tab:       Total Gross Potential Income
      reference: Subsidized Rent
  Operating Statement Detail__TGA: 313 label rows (reference 339), first difference at row 22
      tab:       Total Gross Poten

**Findings**

- **Month headers match in all 90 compared tabs** (30 per file, each against the first tab of its own file). Row 11, columns 1 to 12.
- **Account labels differ in 90 of 90 compared tabs.** Each property has its own account list.
  - Reference tab row counts: `CAL` has 339 label rows in 2023, 337 in 2024, and 331 in 2025.
  - Other tabs range from 267 to 403 label rows (excluding the 4-row tabs below).
  - The first difference is at row 22 or 23 in 41 of 83 tabs. Those rows are the Subsidized Rent block, which CAL has and JHA does not.
  - **Implication:** the unpivot must match rows by GL code (column 0), not by row position.
- **7 tabs have only 4 label rows:**
  - 2023: `IDA`, `LHC`, `LOP`, `NHR`, `HCA` (5 tabs)
  - 2024: `HCA` (1 tab)
  - 2025: `HCA` (1 tab)
  - These tabs contain only the report footer (`Created on: ...`), with no accounts or months. They are empty in the export.

### 1.4 Build Account Map & Ledger Count

Each property has its own account list, so rows can't be matched by position. This step collects every GL account across all tabs and counts how many tabs each one appears in. It also flags any GL code with more than one description, then counts the distinct ledger accounts per property and year.

In [24]:
# Account rows look like "A4010-0100 - Market Rental Income".
# Group 1 is the GL code, group 2 is the description.
gl_pattern = re.compile(r'^([A-Z]\d{4}-\d{4})\s+-\s+(.*)$')

# Collect one record per account row across all files and tabs
records = []
for file_path in t12_files:
    for sheet in pd.ExcelFile(file_path).sheet_names:
        df = pd.read_excel(file_path, sheet_name=sheet, header=None)
        labels = df[0].iloc[12:].dropna().astype(str).str.strip()
        account_count = 0
        for label in labels:
            m = gl_pattern.match(label)
            if m:
                account_count += 1
                records.append({
                    'file': file_path.name,
                    'tab': sheet,
                    'gl': m.group(1),
                    'description': m.group(2),
                })
        # Tabs with very few account rows are listed so they can be checked
        if account_count < 10:
            print(f"Few accounts: {file_path.name} | {sheet} | {account_count} GL rows")

accounts = pd.DataFrame(records)
# A tab is identified by its file and sheet together, since sheet names repeat across files
accounts['tab_key'] = accounts['file'] + ' | ' + accounts['tab']
total_tabs = accounts['tab_key'].nunique()

# One row per GL code: how many tabs use it and how many descriptions it has
gl_summary = accounts.groupby('gl').agg(
    tabs_using=('tab_key', 'nunique'),
    descriptions=('description', 'nunique'),
    example=('description', 'first'),
).sort_index()

print()
print(f"Tabs with account rows: {total_tabs}")
print(f"Distinct GL codes: {len(gl_summary)}")
print(f"GL codes in every tab: {(gl_summary['tabs_using'] == total_tabs).sum()}")
print(f"GL codes with more than one description: {(gl_summary['descriptions'] > 1).sum()}")
print()

# Number each GL code so the last number is the count of distinct GL codes
gl_numbered = gl_summary.reset_index()
gl_numbered.insert(0, 'No.', range(1, len(gl_numbered) + 1))
print(gl_numbered.to_string(index=False))
print()

# Map each file to its year using the "As of Date" value in row 8, column 1 (e.g., 12/31/2023)
file_years = {}
for file_path in t12_files:
    first_sheet = pd.ExcelFile(file_path).sheet_names[0]
    as_of = pd.read_excel(file_path, sheet_name=first_sheet, header=None, nrows=9).iloc[8, 1]
    file_years[file_path.name] = pd.to_datetime(as_of).year

# Add year and property code to each account row (property code is the part after "__")
accounts['year'] = accounts['file'].map(file_years)
accounts['property'] = accounts['tab'].str.split('__').str[-1]

# Count distinct GL codes per property and year
ledger_counts = (
    accounts.groupby(['property', 'year'])['gl']
    .nunique()
    .unstack('year')
)

# Include every property, even those with no GL rows (shown as 0 so they stand out)
all_properties = sorted({s.split('__')[-1] for f in t12_files for s in pd.ExcelFile(f).sheet_names})
ledger_counts = ledger_counts.reindex(all_properties).fillna(0).astype(int)
ledger_counts.columns = [str(c) for c in ledger_counts.columns]

# Add a total row for each year
ledger_counts.loc['Total'] = ledger_counts.sum()

# Print the table with borders: property names left-aligned, counts right-aligned
def print_grid(df):
    headers = [df.index.name or 'Property'] + [str(c) for c in df.columns]
    rows = [[str(idx)] + [str(v) for v in vals] for idx, vals in zip(df.index, df.values)]

    # Each column is as wide as its longest value
    widths = [max(len(cell) for cell in col) for col in zip(headers, *rows)]
    border = '+' + '+'.join('-' * (w + 2) for w in widths) + '+'

    def fmt(cells):
        parts = [f' {cells[0]:<{widths[0]}} '] + [f' {c:>{w}} ' for c, w in zip(cells[1:], widths[1:])]
        return '|' + '|'.join(parts) + '|'

    print(border)
    print(fmt(headers))
    print(border)
    for row in rows:
        # Separate the Total row from the properties
        if row[0] == 'Total':
            print(border)
        print(fmt(row))
    print(border)

print("Ledger accounts (distinct GL codes) by property and year")
print()
print_grid(ledger_counts)

Few accounts: 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-2.xls | Operating Statement Detail__IDA | 0 GL rows
Few accounts: 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-2.xls | Operating Statement Detail__LHC | 0 GL rows
Few accounts: 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-2.xls | Operating Statement Detail__LOP | 0 GL rows
Few accounts: 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-2.xls | Operating Statement Detail__NHR | 0 GL rows
Few accounts: 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-2.xls | Operating Statement Detail__HCA | 0 GL rows
Few accounts: 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties-3.xls | Operating Statement Detail__HCA | 0 GL rows
Few accounts: 2_by Separate tabs - MF OSD - Expand T12 - MULTI--Multifamily Properties.xls | Operating Statement Detail__HCA | 0 GL rows

Tabs with account rows: 86
D